# Project 1 — Sales Performance Analysis

**Goal.** Profile two years of multi-channel e-commerce orders and surface the trends, products, regions, and customers that drive revenue and margin.

**Inputs.** `customers.csv`, `products.csv`, `orders.csv`, `order_items.csv`, `daily_funnel.csv` in `../data/` (synthetic — generated by `../scripts/generate_data.py`).

**Outputs.** Cleaned analytical frames + the executive dashboard rendered by `../dashboard/build_dashboard.py`.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.float_format = '{:,.2f}'.format
DATA = Path('../data')

## 2. Load and inspect

In [ ]:
customers   = pd.read_csv(DATA / 'customers.csv',   parse_dates=['signup_date'])
products    = pd.read_csv(DATA / 'products.csv')
orders      = pd.read_csv(DATA / 'orders.csv',      parse_dates=['order_date'])
order_items = pd.read_csv(DATA / 'order_items.csv')
funnel      = pd.read_csv(DATA / 'daily_funnel.csv', parse_dates=['date'])

for name, df in [('customers', customers), ('products', products),
                 ('orders', orders), ('order_items', order_items),
                 ('funnel', funnel)]:
    print(f'{name:12s} {df.shape[0]:>8,} rows  x  {df.shape[1]} cols')

## 3. Data quality checks

Confirm referential integrity, missing values, and duplicates before downstream analysis.

In [ ]:
print('Missing customer FKs in orders:',
      (~orders['customer_id'].isin(customers['customer_id'])).sum())
print('Missing product FKs in items: ',
      (~order_items['product_id'].isin(products['product_id'])).sum())
print('Duplicate order_ids:           ', orders['order_id'].duplicated().sum())
print('Negative quantities:           ', (order_items['quantity'] <= 0).sum())
print('Negative revenue lines:        ', (order_items['gross_revenue'] < 0).sum())

## 4. Build the analytical fact table

Join order_items with orders, products, and customers, then drop cancelled orders so revenue calculations match the executive definition ("net of cancellations").

In [ ]:
active = orders.query("status != 'Cancelled'")
fact = (order_items
        .merge(active[['order_id', 'order_date', 'customer_id', 'channel', 'status']],
               on='order_id')
        .merge(products[['product_id', 'product_name', 'category']], on='product_id')
        .merge(customers[['customer_id', 'region', 'country', 'segment']],
               on='customer_id'))
fact.head()

## 5. Headline KPIs

In [ ]:
kpis = {
    'Net revenue (USD)'  : fact['gross_revenue'].sum(),
    'Gross profit (USD)' : fact['gross_profit'].sum(),
    'Gross margin (%)'   : fact['gross_profit'].sum() / fact['gross_revenue'].sum() * 100,
    'AOV (USD)'          : fact.groupby('order_id')['gross_revenue'].sum().mean(),
    'Orders'             : fact['order_id'].nunique(),
    'Active customers'   : fact['customer_id'].nunique(),
    'Return rate (%)'    : (orders['status'] == 'Returned').mean() * 100,
}
pd.Series(kpis).to_frame('value')

## 6. Revenue trend

In [ ]:
monthly = (fact
           .assign(month=fact['order_date'].dt.to_period('M').dt.to_timestamp())
           .groupby('month')
           .agg(revenue=('gross_revenue', 'sum'),
                profit =('gross_profit',  'sum'))
           .reset_index())
monthly['mom_pct'] = monthly['revenue'].pct_change() * 100
monthly.tail(12)

In [ ]:
px.bar(monthly, x='month', y='revenue', title='Monthly net revenue').show()

## 7. Category and product mix

In [ ]:
cat = (fact.groupby('category')
            .agg(revenue=('gross_revenue', 'sum'),
                 profit =('gross_profit',  'sum'),
                 units  =('quantity',      'sum'))
            .assign(margin_pct=lambda d: d['profit'] / d['revenue'] * 100)
            .sort_values('revenue', ascending=False))
cat

In [ ]:
top10 = (fact.groupby(['product_name', 'category'])
              ['gross_revenue'].sum()
              .sort_values(ascending=False)
              .head(10)
              .reset_index())
top10

## 8. Regional split

In [ ]:
region = (fact.groupby('region')
               .agg(revenue=('gross_revenue', 'sum'),
                    customers=('customer_id', 'nunique'))
               .assign(share_pct=lambda d: d['revenue'] / d['revenue'].sum() * 100)
               .sort_values('revenue', ascending=False))
region

## 9. Top-customer concentration (Pareto)

In [ ]:
cust_rev = (fact.groupby('customer_id')['gross_revenue']
                 .sum().sort_values(ascending=False))
share_top_20pct = cust_rev.head(int(len(cust_rev) * 0.20)).sum() / cust_rev.sum() * 100
print(f'Top 20% of customers contribute {share_top_20pct:.1f}% of revenue')

## 10. Funnel conversion by channel

In [ ]:
f = (funnel.groupby('channel')[['sessions', 'add_to_cart', 'checkout', 'purchases']]
             .sum()
             .assign(overall_conv_pct=lambda d: d['purchases'] / d['sessions'] * 100,
                     atc_rate_pct    =lambda d: d['add_to_cart'] / d['sessions'] * 100,
                     checkout_rate_pct=lambda d: d['checkout'] / d['add_to_cart'] * 100,
                     purchase_rate_pct=lambda d: d['purchases'] / d['checkout'] * 100)
             .sort_values('purchases', ascending=False))
f

## 11. Insights summary

See `../report.md` for the executive write-up of findings and recommendations.
Run `../dashboard/build_dashboard.py` to render the interactive HTML dashboard.